# Talent Intelligence Agent
**Author:** Aditi Khare | **Dataset:** LinkedIn Job Postings (123K+ postings) | **Stack:** Python, FAISS, LangGraph, flan-t5, MLflow

> An AI-powered agent that answers natural language questions about the job market
> using RAG (Retrieval-Augmented Generation) — no API key required.


## 1. Setup & Dependencies


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


In [ ]:
!pip install langchain langgraph sentence-transformers faiss-cpu \
             transformers mlflow kaggle pandas plotly streamlit \
             langchain-community -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93

## 2. Data Ingestion (Kaggle API)


In [ ]:
from google.colab import userdata
import os

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("Kaggle API ready ✅")




Kaggle API ready ✅


In [ ]:
os.makedirs('/content/data', exist_ok=True)

!kaggle datasets download -d arshkon/linkedin-job-postings \
    --unzip -p /content/data/

print(os.listdir('/content/data/'))


Dataset URL: https://www.kaggle.com/datasets/arshkon/linkedin-job-postings
License(s): CC-BY-SA-4.0
 78% 124M/159M [00:00<00:00, 1.27GB/s]
100% 159M/159M [00:00<00:00, 944MB/s] 
['companies', 'postings.csv', 'jobs', 'mappings']


## 3. Exploratory Data Analysis (EDA)
> Analyzing 123,849 LinkedIn job postings across 24,428 companies


In [ ]:
import pandas as pd

df = pd.read_csv('/content/data/postings.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
df.head(5)


Shape: (123849, 31)

Columns:
['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

print("=== DATASET OVERVIEW ===")
print(f"Total job postings: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing,
                            'missing_%': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values(
    'missing_%', ascending=False))


=== DATASET OVERVIEW ===
Total job postings: 123,849
Columns: 31

=== MISSING VALUES ===
                            missing_count  missing_%
closed_time                        122776       99.1
skills_desc                        121410       98.0
med_salary                         117569       94.9
remote_allowed                     108603       87.7
applies                            100529       81.2
min_salary                          94056       75.9
max_salary                          94056       75.9
pay_period                          87776       70.9
compensation_type                   87776       70.9
normalized_salary                   87776       70.9
currency                            87776       70.9
posting_domain                      39968       32.3
application_url                     36665       29.6
formatted_experience_level          29409       23.7
fips                                27415       22.1
zip_code                            20872       16.9
views     

In [ ]:
# Top 20 most common job titles
top_titles = df['title'].value_counts().head(20).reset_index()
top_titles.columns = ['title', 'count']

fig = px.bar(top_titles, x='count', y='title', orientation='h',
             title='Top 20 Most Posted Job Titles',
             color='count', color_continuous_scale='Blues',
             height=600)
fig.update_layout(yaxis={'categoryorder': 'total ascending'},
                  showlegend=False)
fig.show()


In [ ]:
# Experience level distribution
exp = df['formatted_experience_level'].value_counts().reset_index()
exp.columns = ['level', 'count']

fig = px.pie(exp, values='count', names='level',
             title='Job Postings by Experience Level',
             color_discrete_sequence=px.colors.sequential.Blues_r)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()


In [ ]:
# Filter rows with salary data
salary_df = df.dropna(subset=['max_salary', 'min_salary']).copy()
salary_df = salary_df[
    (salary_df['max_salary'] < 500000) &   # remove outliers
    (salary_df['min_salary'] > 0)
]
salary_df['avg_salary'] = (salary_df['max_salary'] + salary_df['min_salary']) / 2

print(f"Jobs with salary data: {len(salary_df):,} ({len(salary_df)/len(df)*100:.1f}%)")
print(f"\nSalary Stats:")
print(salary_df['avg_salary'].describe().round(0))

# Salary by experience level
fig = px.box(salary_df, x='formatted_experience_level', y='avg_salary',
             title='Salary Distribution by Experience Level',
             color='formatted_experience_level',
             color_discrete_sequence=px.colors.sequential.Blues_r)
fig.update_layout(showlegend=False, xaxis_title='Experience Level',
                  yaxis_title='Average Salary (USD)')
fig.show()


Jobs with salary data: 29,660 (23.9%)

Salary Stats:
count     29660.0
mean      73160.0
std       70592.0
min           1.0
25%          42.0
50%       71500.0
75%      119700.0
max      450726.0
Name: avg_salary, dtype: float64


In [ ]:
top_companies = df['company_name'].value_counts().head(15).reset_index()
top_companies.columns = ['company', 'postings']

fig = px.bar(top_companies, x='postings', y='company', orientation='h',
             title='Top 15 Most Active Hiring Companies',
             color='postings', color_continuous_scale='Teal',
             height=500)
fig.update_layout(yaxis={'categoryorder': 'total ascending'},
                  showlegend=False)
fig.show()


In [ ]:
# Work type distribution (Remote vs On-site vs Hybrid)
if 'work_type' in df.columns:
    work_type = df['work_type'].value_counts().reset_index()
    work_type.columns = ['type', 'count']
    fig = px.pie(work_type, values='count', names='type',
                 title='Remote vs On-site vs Hybrid Breakdown',
                 color_discrete_sequence=['#1f77b4','#aec7e8','#c6dbef'])
    fig.show()

# Top hiring locations
top_locations = df['location'].value_counts().head(15).reset_index()
top_locations.columns = ['location', 'count']
fig = px.bar(top_locations, x='count', y='location', orientation='h',
             title='Top 15 Hiring Locations',
             color='count', color_continuous_scale='Blues', height=500)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


##  4. Data Cleaning & Filtering
> Narrowing to 1,653 targeted analyst roles (Data, Business, People, Financial)


In [ ]:
import re

skill_keywords = [
    'sql', 'python', 'tableau', 'power bi', 'excel',
    'spark', 'machine learning', 'statistics', 'workday',
    'looker', 'databricks', 'snowflake', 'aws', 'azure',
    'dbt', 'airflow', 'jira', 'salesforce', 'alteryx'
]

# Use word boundaries — avoids single-letter false matches
desc = df['description'].fillna('').str.lower()
skill_counts = {
    skill: desc.str.contains(rf'\b{re.escape(skill)}\b', na=False).sum()
    for skill in skill_keywords
}

skills_df = pd.DataFrame(
    sorted(skill_counts.items(), key=lambda x: x[1], reverse=True),
    columns=['skill', 'count']
)
print(skills_df.head(10))


              skill  count
0             excel  18107
1               sql   5183
2            python   4648
3               aws   3161
4             azure   2918
5        salesforce   2893
6           workday   2469
7        statistics   1956
8  machine learning   1710
9              jira   1538


In [ ]:
print("=" * 50)
print("         EDA SUMMARY — KEY FINDINGS")
print("=" * 50)
print(f"Total postings analyzed:     {len(df):,}")
print(f"Unique companies hiring:     {df['company_name'].nunique():,}")
print(f"Unique locations:            {df['location'].nunique():,}")
print(f"Jobs with salary data:       {len(salary_df):,}")
print(f"Median avg salary:           ${salary_df['avg_salary'].median():,.0f}")
print(f"Most demanded skill:         {skills_df.iloc[0]['skill'].upper()}")
print(f"Top hiring company:          {df['company_name'].value_counts().index[0]}")
print("=" * 50)


         EDA SUMMARY — KEY FINDINGS
Total postings analyzed:     123,849
Unique companies hiring:     24,428
Unique locations:            8,526
Jobs with salary data:       29,660
Median avg salary:           $71,500
Most demanded skill:         EXCEL
Top hiring company:          Liberty Healthcare and Rehabilitation Services


In [ ]:
# Filter to analyst-relevant roles
analyst_roles = df[df['title'].str.contains(
    'analyst|analytics|intelligence|people ops|operations|business intel',
    case=False, na=False
)].copy()

# Drop rows with no description
analyst_roles = analyst_roles.dropna(subset=['description']).reset_index(drop=True)

# Create combined text field for embedding
analyst_roles['embed_text'] = (
    analyst_roles['title'] + " | " +
    analyst_roles['company_name'].fillna('') + " | " +
    analyst_roles['description'].str[:400]
)

print(f"Analyst roles filtered: {len(analyst_roles):,}")
print(f"Sample embed text:\n{analyst_roles['embed_text'].iloc[0]}")

# Save filtered dataset
analyst_roles.to_csv('/content/data/jobs_clean.csv', index=False)
print("\nSaved jobs_clean.csv ✅")


Analyst roles filtered: 7,811
Sample embed text:
Board Certified Behavior Analyst | Kona Medical Consulting | JOB OVERVIEW:
The BCBA will provide support to individuals and their families by coordinating and providing services in Applied Behavior Analysis, function analyses and assessment, behavior acquisition and reduction procedures, and adaptive life skills.
A BCBA will also oversee the programming of associate behavior analysts and provide ongoing support as it relates to the implementation and docume

Saved jobs_clean.csv ✅


In [ ]:
# More precise filter - target data/business/people analytics roles only
analyst_roles = df[
    df['title'].str.contains(
        'data analyst|business analyst|people analyst|hr analyst|'
        'workforce analyst|operations analyst|reporting analyst|'
        'analytics analyst|intelligence analyst|financial analyst|'
        'marketing analyst|product analyst|people operations',
        case=False, na=False
    )
].copy()

# Drop rows with no description
analyst_roles = analyst_roles.dropna(subset=['description']).reset_index(drop=True)

# Create combined text field for embedding
analyst_roles['embed_text'] = (
    analyst_roles['title'] + " | " +
    analyst_roles['company_name'].fillna('') + " | " +
    analyst_roles['description'].str[:400]
)

print(f"Refined analyst roles: {len(analyst_roles):,}")
print(f"\nTop 10 role titles:")
print(analyst_roles['title'].value_counts().head(10))
print(f"\nSample embed text:\n{analyst_roles['embed_text'].iloc[0]}")

analyst_roles.to_csv('/content/data/jobs_clean.csv', index=False)
print("\nSaved ✅")


Refined analyst roles: 1,653

Top 10 role titles:
title
Business Analyst              146
Data Analyst                  137
Financial Analyst             113
Senior Financial Analyst       68
Senior Business Analyst        34
Senior Data Analyst            25
Online Data Analyst            20
Technical Business Analyst     16
Sr. Financial Analyst          15
Sales Operations Analyst       14
Name: count, dtype: int64

Sample embed text:
Technical Business Analyst | CapAcuity | We’re actively seeking a Technical Business Analyst to join our team and help us continue to grow our business. As a Technical Business Analyst, you will be responsible for working with our information technology and business teams to understand our business requirements and translate them into technical specifications. You will also be responsible for working with our development team to ensure t

Saved ✅


In [ ]:
# ── DATA CLEANING ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("=== MISSING VALUES BEFORE CLEANING ===")
print(analyst_roles[['company_name', 'formatted_experience_level',
                      'max_salary', 'min_salary']].isnull().sum())
print(f"Total rows: {len(analyst_roles):,}")

# ── Fix 1: Company Name ────────────────────────────────
analyst_roles['company_name'] = \
    analyst_roles['company_name'].fillna('Unknown Company')

# ── Fix 2: Experience Level ────────────────────────────
analyst_roles['formatted_experience_level'] = \
    analyst_roles['formatted_experience_level'].fillna('Not Specified')

# ── Fix 3: Salary — FLAG only, never impute ────────────
# 75% of postings hide salary intentionally.
# Filling with mean/median would mislead job seekers.
# Instead: flag presence and create a display column.
analyst_roles['has_salary_data'] = analyst_roles['max_salary'].notna()

analyst_roles['salary_display'] = analyst_roles.apply(
    lambda r: f"${int(r['min_salary']):,} – ${int(r['max_salary']):,}"
    if pd.notna(r['min_salary']) and pd.notna(r['max_salary'])
    and r['max_salary'] > 1000
    else "Not disclosed", axis=1
)

# ── Fix 4: Drop rows with no job description ───────────
# Description is required for FAISS embeddings
before = len(analyst_roles)
analyst_roles = analyst_roles.dropna(subset=['description'])
dropped = before - len(analyst_roles)

# ── Verify ─────────────────────────────────────────────
print("\n=== MISSING VALUES AFTER CLEANING ===")
print(analyst_roles[['company_name', 'formatted_experience_level',
                      'has_salary_data']].isnull().sum())
print(f"\nJobs with salary disclosed : {analyst_roles['has_salary_data'].sum():,} "
      f"({analyst_roles['has_salary_data'].mean()*100:.1f}%)")
print(f"Jobs with salary hidden    : {(~analyst_roles['has_salary_data']).sum():,}")
print(f"Rows dropped (no desc)     : {dropped}")
print(f"\n✅ Clean dataset ready     : {len(analyst_roles):,} rows")


=== MISSING VALUES BEFORE CLEANING ===
company_name                    15
formatted_experience_level     405
max_salary                    1159
min_salary                    1159
dtype: int64
Total rows: 1,653

=== MISSING VALUES AFTER CLEANING ===
company_name                  0
formatted_experience_level    0
has_salary_data               0
dtype: int64

Jobs with salary disclosed : 494 (29.9%)
Jobs with salary hidden    : 1,159
Rows dropped (no desc)     : 0

✅ Clean dataset ready     : 1,653 rows


## 5. FAISS Vector Index
> Embedding job descriptions using all-MiniLM-L6-v2 (384 dimensions)


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')  # fast, free, 80MB

texts = analyst_roles['embed_text'].tolist()

print(f"Encoding {len(texts):,} job postings...")
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64
)
embeddings = np.array(embeddings).astype('float32')

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save everything
faiss.write_index(index, '/content/data/jobs.index')
with open('/content/data/embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

print(f"\n✅ FAISS index built!")
print(f"   Vectors indexed: {index.ntotal:,}")
print(f"   Embedding dimensions: {dimension}")


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 1,653 job postings...


Batches:   0%|          | 0/26 [00:00<?, ?it/s]


✅ FAISS index built!
   Vectors indexed: 1,653
   Embedding dimensions: 384


In [ ]:
def semantic_search(query: str, top_k: int = 5) -> pd.DataFrame:
    """Encode query → search FAISS → return top matching jobs"""
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k)

    results = analyst_roles.iloc[indices[0]].copy()
    # Convert L2 distance to similarity score (0-1, higher = better)
    results['similarity_score'] = (1 / (1 + distances[0])).round(3)

    return results[['title', 'company_name', 'location',
                     'formatted_experience_level',
                     'similarity_score']].reset_index(drop=True)

# ---- TEST QUERIES ----
print("=" * 55)
print("TEST 1: People Analytics & Workforce Planning")
print("=" * 55)
print(semantic_search("people analytics workforce planning HR"))

print("\n" + "=" * 55)
print("TEST 2: SQL Python Data Analyst")
print("=" * 55)
print(semantic_search("data analyst SQL python tableau dashboard"))

print("\n" + "=" * 55)
print("TEST 3: Business Analyst Stakeholder Management")
print("=" * 55)
print(semantic_search("business analyst requirements gathering stakeholders"))


TEST 1: People Analytics & Workforce Planning
                                               title  \
0             Workforce Metrics Intelligence Analyst   
1                                         HR Analyst   
2                          HRIS Analyst / HR Analyst   
3  Workforce Analyst (Job Analysis and Hiring Ass...   
4                   Human Resources Business Analyst   

                       company_name                location  \
0                            Oracle           United States   
1      Ascend Performance Materials             Houston, TX   
2                   Sharp Decisions               Tampa, FL   
3  FYI - For Your Information, Inc.          Washington, DC   
4                       InfoStride   San Francisco Bay Area   

  formatted_experience_level  similarity_score  
0           Mid-Senior level             0.573  
1           Mid-Senior level             0.552  
2                  Associate             0.546  
3           Mid-Senior level             0

##6. LLM Configuration
> Running in LOCAL mode using google/flan-t5-base — no Gemini quota needed


In [ ]:
MODE = "local"

if MODE == "local":
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    import torch

    print("Loading flan-t5-base...")
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
    llm_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    llm_model = llm_model.to(device)

    def ask_llm(prompt: str) -> str:
        inputs = tokenizer(
            prompt[:1000],
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=False
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"✅ Local LLM ready | Device: {device}")

elif MODE == "gemini":
    from google.colab import userdata
    import google.generativeai as genai
    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
    gemini_model = genai.GenerativeModel('gemini-1.5-flash')
    def ask_llm(prompt: str) -> str:
        return gemini_model.generate_content(prompt).text
    print("✅ Gemini LLM ready")

print(f"Running in {MODE.upper()} mode")


Loading flan-t5-base...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Local LLM ready | Device: cuda
Running in LOCAL mode


## 7. LangGraph RAG Agent
> Retrieve → Generate pipeline using LangGraph StateGraph


In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, List

# Define agent state
class AgentState(TypedDict):
    messages: List
    search_results: str
    final_answer: str

# Node 1: Retrieve relevant jobs from FAISS
def retrieve_node(state: AgentState) -> AgentState:
    query = state['messages'][-1].content
    results = semantic_search(query, top_k=5)
    state['search_results'] = results.to_string(index=False)
    return state

# Node 2: Generate insight using LLM + retrieved context
def generate_node(state: AgentState) -> AgentState:
    query = state['messages'][-1].content
    context = state['search_results']

    prompt = f"""You are a talent intelligence analyst helping job seekers.

Job postings retrieved:
{context[:600]}

Question: {query}

Answer in 2-3 sentences with specific insights:"""

    answer = ask_llm(prompt)
    state['final_answer'] = answer
    state['messages'].append(AIMessage(content=answer))
    return state

# Build the graph
graph = StateGraph(AgentState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
graph.set_entry_point("retrieve")
agent = graph.compile()

def run_agent(question: str) -> dict:
    result = agent.invoke({
        "messages": [HumanMessage(content=question)],
        "search_results": "",
        "final_answer": ""
    })
    return {
        "question": question,
        "retrieved_jobs": result['search_results'],
        "answer": result['final_answer']
    }

print("✅ LangGraph agent compiled and ready!")


✅ LangGraph agent compiled and ready!


## 8. Agent Q&A Demo


In [ ]:
import re

def run_agent(question: str) -> dict:
    q = question.lower()

    # Route to the right answer function based on question type
    if any(w in q for w in ['skill', 'require', 'need', 'qualif']):
        answer = answer_skills_question(q)
    elif any(w in q for w in ['salary', 'pay', 'compensat', 'earn']):
        answer = answer_salary_question(q)
    elif any(w in q for w in ['compan', 'who is hiring', 'employer']):
        answer = answer_company_question(q)
    elif any(w in q for w in ['location', 'city', 'where', 'state']):
        answer = answer_location_question(q)
    elif any(w in q for w in ['experience', 'level', 'senior', 'junior']):
        answer = answer_experience_question(q)
    else:
        # Fallback: semantic search + return top results
        results = semantic_search(question, top_k=5)
        answer = f"Top matching roles: {', '.join(results['title'].tolist())}"

    return {"question": question, "answer": answer}


def answer_skills_question(query: str) -> str:
    # Detect role from query
    role_map = {
        'people analyst': 'people analyst|hr analyst|workforce analyst',
        'data analyst': 'data analyst',
        'business analyst': 'business analyst',
        'financial analyst': 'financial analyst'
    }
    role_filter = None
    for role, pattern in role_map.items():
        if role in query:
            role_filter = pattern
            break

    skill_keywords = ['sql', 'python', 'tableau', 'power bi', 'excel',
                      'machine learning', 'statistics', 'workday', 'looker',
                      'databricks', 'snowflake', 'aws', 'azure', 'jira']

    data = analyst_roles if not role_filter else \
           analyst_roles[analyst_roles['title'].str.contains(
               role_filter, case=False, na=False)]

    desc = data['description'].fillna('').str.lower()
    counts = {s: desc.str.contains(rf'\b{s}\b', na=False).sum()
              for s in skill_keywords}
    top_skills = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:5]
    skills_str = ', '.join([s[0].upper() for s in top_skills])

    role_name = role_filter.split('|')[0].title() if role_filter else "Analyst"
    return f"Top 5 skills for {role_name} roles: {skills_str}. " \
           f"Based on {len(data):,} job postings analyzed."


def answer_salary_question(query: str) -> str:
    salary_df = analyst_roles.dropna(subset=['max_salary', 'min_salary']).copy()
    salary_df = salary_df[
        (salary_df['max_salary'] > 1000) &
        (salary_df['max_salary'] < 500000)
    ]

    # Detect role
    for role in ['data analyst', 'business analyst', 'financial analyst',
                 'people analyst']:
        if role in query:
            salary_df = salary_df[salary_df['title'].str.contains(
                role, case=False, na=False)]
            break

    if len(salary_df) == 0:
        return "Insufficient salary data for this specific role in the dataset."

    salary_df['avg'] = (salary_df['max_salary'] + salary_df['min_salary']) / 2
    low = int(salary_df['min_salary'].median())
    high = int(salary_df['max_salary'].median())
    med = int(salary_df['avg'].median())

    return f"Salary range: ${low:,} – ${high:,} (median ~${med:,}/year). " \
           f"Based on {len(salary_df):,} postings with salary data."


def answer_company_question(query: str) -> str:
    data = analyst_roles.copy()
    if 'remote' in query:
        data = data[data['location'].str.contains(
            'remote|united states', case=False, na=False)]
    top = data['company_name'].value_counts().head(5).reset_index()
    companies = ', '.join(top['company_name'].dropna().tolist())
    return f"Top hiring companies: {companies}."


def answer_location_question(query: str) -> str:
    top_locs = analyst_roles['location'].value_counts().head(5).reset_index()
    locs = ', '.join(top_locs['location'].dropna().tolist())
    return f"Top locations for analyst jobs: {locs}."


def answer_experience_question(query: str) -> str:
    exp = analyst_roles['formatted_experience_level'].value_counts()
    top = exp.head(3)
    result = ', '.join([f"{k} ({v} roles)" for k, v in top.items()])
    return f"Experience level breakdown: {result}."


# ---- TEST ----
questions = [
    "What skills do I need for a People Analyst role?",
    "What companies are hiring business analysts remotely?",
    "What is the salary range for a data analyst?",
    "What experience level is required for data analyst roles?",
    "Which locations have the most business analyst jobs?"
]

for q in questions:
    result = run_agent(q)
    print("=" * 60)
    print(f"❓ {result['question']}")
    print(f"🤖 {result['answer']}")
    print()


❓ What skills do I need for a People Analyst role?
🤖 Top 5 skills for People Analyst roles: EXCEL, WORKDAY, SQL, POWER BI, STATISTICS. Based on 8 job postings analyzed.

❓ What companies are hiring business analysts remotely?
🤖 Top hiring companies: ATC, Talentify.io, Unknown Company, Insight Global, Datasys Consulting and Software Inc..

❓ What is the salary range for a data analyst?
🤖 Salary range: $85,000 – $110,125 (median ~$95,000/year). Based on 64 postings with salary data.

❓ What experience level is required for data analyst roles?
🤖 Top 5 skills for Data Analyst roles: SQL, EXCEL, PYTHON, STATISTICS, TABLEAU. Based on 408 job postings analyzed.

❓ Which locations have the most business analyst jobs?
🤖 Top locations for analyst jobs: United States, New York, NY, Chicago, IL, Atlanta, GA, Austin, TX.



## 9. MLflow Experiment Tracking


In [ ]:
import mlflow

# Better prompt for cleaner answers
def run_agent(question: str) -> dict:
    result = agent.invoke({
        "messages": [HumanMessage(content=question)],
        "search_results": "",
        "final_answer": ""
    })
    return {
        "question": question,
        "retrieved_jobs": result['search_results'],
        "answer": result['final_answer']
    }

# MLflow experiment tracking
mlflow.set_experiment("talent_intelligence_agent")

test_queries = [
    "What skills do I need for a People Analyst role?",
    "What companies are hiring business analysts remotely?",
    "What is the experience level required for data analyst roles?",
    "Which locations have the most business analyst jobs?"
]

with mlflow.start_run(run_name="rag_pipeline_v1"):
    mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
    mlflow.log_param("llm_model", "google/flan-t5-base")
    mlflow.log_param("llm_mode", MODE)
    mlflow.log_param("index_size", index.ntotal)
    mlflow.log_param("top_k", 5)
    mlflow.log_param("dataset", "linkedin-job-postings")
    mlflow.log_param("filtered_roles", len(analyst_roles))

    print("Running evaluation queries...\n")
    for i, query in enumerate(test_queries):
        results = semantic_search(query, top_k=5)
        avg_score = results['similarity_score'].mean()
        mlflow.log_metric(f"avg_similarity_q{i+1}", round(avg_score, 4))
        print(f"Q{i+1}: {query[:50]}...")
        print(f"     Avg similarity: {avg_score:.4f}\n")

    mlflow.log_param("total_queries_tested", len(test_queries))
    print("✅ MLflow run logged!")
    print("\nRun this to view MLflow UI:")
    print("  mlflow ui --port 5000")


2026/03/13 02:40:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/13 02:40:25 INFO mlflow.store.db.utils: Updating database tables
2026/03/13 02:40:26 INFO mlflow.tracking.fluent: Experiment with name 'talent_intelligence_agent' does not exist. Creating a new experiment.


Running evaluation queries...

Q1: What skills do I need for a People Analyst role?...
     Avg similarity: 0.5378

Q2: What companies are hiring business analysts remote...
     Avg similarity: 0.5958

Q3: What is the experience level required for data ana...
     Avg similarity: 0.5818

Q4: Which locations have the most business analyst job...
     Avg similarity: 0.5362

✅ MLflow run logged!

Run this to view MLflow UI:
  mlflow ui --port 5000


## 10. Skills Gap Analyzer


In [ ]:
def skills_gap_analyzer(user_skills: list, target_role: str) -> None:
    """Compare your skills vs what the market demands for a role"""

    role_jobs = analyst_roles[analyst_roles['title'].str.contains(
        target_role, case=False, na=False)]

    skill_keywords = ['sql', 'python', 'tableau', 'power bi', 'excel',
                      'machine learning', 'statistics', 'workday', 'looker',
                      'databricks', 'snowflake', 'aws', 'azure', 'jira',
                      'alteryx', 'sap', 'r']

    desc = role_jobs['description'].fillna('').str.lower()
    market_skills = {s: desc.str.contains(rf'\b{s}\b', na=False).sum()
                     for s in skill_keywords}
    top_market = sorted(market_skills.items(),
                        key=lambda x: x[1], reverse=True)[:10]
    top_market_set = set([s[0] for s in top_market])
    user_set = set([s.lower() for s in user_skills])

    have = user_set & top_market_set
    gaps = top_market_set - user_set

    print(f"\n{'='*55}")
    print(f"  SKILLS GAP ANALYSIS: {target_role.upper()}")
    print(f"  Based on {len(role_jobs):,} job postings")
    print(f"{'='*55}")
    print(f"✅ Skills you HAVE:    {', '.join(have).upper() or 'None matched'}")
    print(f"❌ Skills you're MISSING: {', '.join(gaps).upper()}")
    print(f"📊 Match Score:        {len(have)}/{len(top_market_set)}")

    # Aditi's actual skills from her CV
aditi_skills = ['sql', 'python', 'tableau', 'power bi', 'excel',
                'workday', 'statistics', 'jira']

skills_gap_analyzer(aditi_skills, "Data Analyst")
skills_gap_analyzer(aditi_skills, "Business Analyst")
skills_gap_analyzer(aditi_skills, "People Analyst")



  SKILLS GAP ANALYSIS: DATA ANALYST
  Based on 408 job postings
✅ Skills you HAVE:    STATISTICS, TABLEAU, EXCEL, PYTHON, SQL, POWER BI
❌ Skills you're MISSING: SNOWFLAKE, R, MACHINE LEARNING, AWS
📊 Match Score:        6/10

  SKILLS GAP ANALYSIS: BUSINESS ANALYST
  Based on 556 job postings
✅ Skills you HAVE:    STATISTICS, TABLEAU, EXCEL, PYTHON, JIRA, SQL, POWER BI
❌ Skills you're MISSING: AZURE, SAP, AWS
📊 Match Score:        7/10

  SKILLS GAP ANALYSIS: PEOPLE ANALYST
  Based on 1 job postings
✅ Skills you HAVE:    STATISTICS, TABLEAU, EXCEL, PYTHON, SQL, POWER BI, WORKDAY
❌ Skills you're MISSING: LOOKER, MACHINE LEARNING, DATABRICKS
📊 Match Score:        7/10


In [ ]:
from google.colab import files

# Save clean dataset
analyst_roles.to_csv('jobs_clean.csv', index=False)

# Save requirements
with open('requirements.txt', 'w') as f:
    f.write("""langchain
langgraph
langchain-community
sentence-transformers
faiss-cpu
transformers
mlflow
kaggle
pandas
plotly
streamlit
torch
scikit-learn
""")

# Download all files
files.download('jobs_clean.csv')
files.download('requirements.txt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>